# Context Strategy Evaluation

Compares context strategies on LongBench using Gemini 2.5 Flash as the answer model and judge.

**Strategies:** `full_context`, `retrieval`, `compression`, `gemini_summarization`, `retrieval_compression`

**Task types:** QA (`qasper`, `hotpotqa`, `passage_count`) and Summarization (`gov_report`, `multi_news`)

QA and Summarization examples are balanced by running more examples per summarization task to match total QA count.

## 1. Authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from huggingface_hub import login
login()

## 2. Clone Repo

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/rissingh23/Context-Opt.git'
REPO_DIR = Path('/content/Context-Opt')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Config

**Task type balancing:**
- 3 QA tasks × `LIMIT_QA` examples = total QA examples
- 2 Summarization tasks × `LIMIT_SUMM` examples = total Summarization examples
- Default: 50 QA each (150 total) and 75 Summarization each (150 total) → balanced

**Judge:** set `USE_JUDGE = False` to skip LLM scoring and only use automatic metrics (faster, no extra API calls).

In [ ]:
GCP_PROJECT  = 'contextops-498110'
GCP_LOCATION = 'us-central1'
GEMINI_MODEL = 'gemini-2.5-flash'

# Examples per task — balanced across task types
# 3 QA tasks × LIMIT_QA = 3 × 50 = 150 QA examples
# 2 Summ tasks × LIMIT_SUMM = 2 × 75 = 150 Summarization examples
LIMIT_QA   = 50   # qasper, hotpotqa, passage_count
LIMIT_SUMM = 75   # gov_report, multi_news

# Utility function: Utility = Quality - λ·Cost - β·Latency
LAMBDA_COST  = 100.0   # penalty per dollar
BETA_LATENCY = 0.001   # penalty per second

# Judge: True = Gemini scores each answer 0-1 (better quality metric, costs extra API calls)
USE_JUDGE = True

STRATEGIES = 'full_context retrieval compression gemini_summarization retrieval_compression'

JUDGE_FLAGS = f'--judge vertexai --judge-model {GEMINI_MODEL} --quality-source llm_judge' if USE_JUDGE else '--judge disabled --quality-source automatic'

print(f'QA examples: 3 tasks × {LIMIT_QA} = {3 * LIMIT_QA}')
print(f'Summarization examples: 2 tasks × {LIMIT_SUMM} = {2 * LIMIT_SUMM}')
print(f'Judge: {"Gemini" if USE_JUDGE else "disabled"}')
print(f'Utility = Quality - {LAMBDA_COST}·Cost - {BETA_LATENCY}·Latency')

In [ ]:
# Verify Gemini connection
from google import genai
client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
response = client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK.')
print('Gemini connection:', response.text)

## 5. Download LongBench Data

Downloads all 200 test examples per task. Only needs to run once per session.

In [ ]:
!python3 -m src.data.download_longbench \
  --tasks qasper hotpotqa passage_count gov_report multi_news

## 6. Smoke Test — 1 Example

Run this first to confirm all strategies work before the full eval.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper \
  --strategies {STRATEGIES} \
  --limit 1 \
  --provider vertexai \
  --model {GEMINI_MODEL} \
  {JUDGE_FLAGS} \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --rows-output outputs/processed/smoke_rows.csv \
  --aggregate-output outputs/processed/smoke_summary.csv \
  --json-output outputs/processed/smoke_checkpoint.jsonl

In [ ]:
import pandas as pd
pd.read_csv('outputs/processed/smoke_summary.csv')[[
    'task', 'strategy', 'num_examples', 'error_rate',
    'avg_quality_score', 'avg_compression_ratio', 'avg_total_latency_sec'
]]

## 7. Full Eval — QA Tasks

Runs `qasper`, `hotpotqa`, `passage_count` with `LIMIT_QA` examples each.

**Checkpoint/resume:** results are saved after every example. If interrupted, rerun this cell — it skips already-completed rows automatically.

**Expected warnings (all harmless):**
- `Token indices sequence length is longer than...` — LLMLingua truncates long docs internally
- `FutureWarning: _check_is_size` — bitsandbytes internal warning
- `The following generation flags are not valid` — Gemini config warning

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper hotpotqa passage_count \
  --strategies {STRATEGIES} \
  --limit {LIMIT_QA} \
  --provider vertexai \
  --model {GEMINI_MODEL} \
  {JUDGE_FLAGS} \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --lambda-cost {LAMBDA_COST} \
  --beta-latency {BETA_LATENCY} \
  --rows-output outputs/processed/eval_rows.csv \
  --aggregate-output outputs/processed/eval_summary.csv \
  --json-output outputs/processed/eval_checkpoint.jsonl

## 8. Full Eval — Summarization Tasks

Runs `gov_report`, `multi_news` with `LIMIT_SUMM` examples each (more per task to balance total count with QA).

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks gov_report multi_news \
  --strategies {STRATEGIES} \
  --limit {LIMIT_SUMM} \
  --provider vertexai \
  --model {GEMINI_MODEL} \
  {JUDGE_FLAGS} \
  --vertexai-project {GCP_PROJECT} \
  --vertexai-location {GCP_LOCATION} \
  --lambda-cost {LAMBDA_COST} \
  --beta-latency {BETA_LATENCY} \
  --rows-output outputs/processed/eval_rows.csv \
  --aggregate-output outputs/processed/eval_summary.csv \
  --json-output outputs/processed/eval_checkpoint.jsonl

## 9. View Results

In [ ]:
import pandas as pd

df = pd.read_csv('outputs/processed/eval_summary.csv')
df[[
    'task', 'task_type', 'strategy',
    'num_examples', 'error_rate',
    'avg_llm_judge_score', 'avg_quality_score',
    'avg_rouge_l', 'avg_token_f1',
    'avg_compression_ratio',
    'avg_total_latency_sec',
    'avg_estimated_cost',
    'avg_utility_score',
]].sort_values(['task', 'avg_utility_score'], ascending=[True, False])

In [ ]:
# Task type summary — compare QA vs Summarization
rows = pd.read_csv('outputs/processed/eval_rows.csv')
print('Examples per task type:')
print(rows.groupby('task_type')['example_id'].count())
print()
rows.groupby(['task_type', 'strategy']).agg(
    avg_quality=('quality_score', 'mean'),
    avg_compression=('compression_ratio', 'mean'),
    avg_utility=('utility_score', 'mean'),
).sort_values(['task_type', 'avg_utility'], ascending=[True, False])

In [ ]:
# Check for errors
errors = rows[rows['error'].fillna('') != '']
print(f'Error rows: {len(errors)} / {len(rows)}')
if len(errors):
    print(errors[['task', 'strategy', 'error']].to_string())

## 10. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = Path('/content/drive/MyDrive/context-opt-results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

!cp outputs/processed/eval_rows.csv {DRIVE_OUT}/eval_rows.csv
!cp outputs/processed/eval_summary.csv {DRIVE_OUT}/eval_summary.csv
!cp outputs/processed/eval_checkpoint.jsonl {DRIVE_OUT}/eval_checkpoint.jsonl

print(f'Saved to {DRIVE_OUT}')